# Checkout Redesign A/B Test: End-to-End Case Study

An ecommerce site redesigned its checkout flow. We ran a 2-week experiment on
80,000 users (50/50 control vs treatment) to decide: **ship or not?**

This notebook walks through the full analyst loop: design, sizing, validity
checks, the primary read, guardrails, variance reduction (CUPED), segment
analysis, novelty effects, and the dollar-denominated business decision.

Dataset: synthetic, generated with a fixed seed (`python src/generate.py`).

## 0. Load the data

`src/generate.py` builds `data/experiment.csv` (80,000 rows, seed 20261013).
The analysis logic lives in `src/analyze.py`; this notebook shows the same
steps interactively.

In [ ]:
import pandas as pd
df = pd.read_csv("data/experiment.csv")
print(df.shape)
print(df.head(3).to_string())
print("\nArm sizes:")
print(df["arm"].value_counts())

(80000, 7)
   user_id        arm  device  day  prior_buyer  converted  order_value
0        1  treatment  mobile    4            0          0         0.00
1        2  treatment  mobile   13            1          0         0.00
2        3    control  mobile   13            0          0         0.00

Arm sizes:
arm
control      40000
treatment    40000
Name: count, dtype: int64

## 1. Design and power analysis

Before looking at results: was the test big enough to detect what we care
about? With a 12% baseline and a +1.5pp minimum detectable effect at 80%
power, the two-proportion z-test needs ~7.8k users per arm. We ran 40k per
arm, deliberately overpowered so the device-segment analysis later has real
statistical teeth.

In [ ]:
import sys; sys.path.insert(0, "src")
from analyze import required_n_per_arm
print(f"Required per arm for +1.5pp MDE at 80% power: {required_n_per_arm(0.12, 0.015):,}")

Required per arm for +1.5pp MDE at 80% power: 7,761

## 2. Validity check: sample ratio mismatch

A broken randomization invalidates everything downstream, so check it first.
Exact 40,000/40,000 split, chi-square p = 1.0000. Clean.

In [ ]:
from scipy import stats
n_c = (df.arm == "control").sum(); n_t = (df.arm == "treatment").sum()
_, p_srm = stats.chisquare([n_c, n_t])
print(f"control={n_c:,}, treatment={n_t:,}, SRM chi-square p={p_srm:.4f}")

control=40,000, treatment=40,000, SRM chi-square p=1.0000

## 3. Primary metric: conversion rate

Two-proportion z-test with a 95% confidence interval on the absolute lift.

In [ ]:
import sys; sys.path.insert(0, "src")
from analyze import two_proportion_test, fmt_p
ctrl = df[df.arm == "control"]; treat = df[df.arm == "treatment"]
x_c, x_t = ctrl.converted.sum(), treat.converted.sum()
diff, z, p, lo, hi = two_proportion_test(x_c, len(ctrl), x_t, len(treat))
print(f"control:   {x_c/len(ctrl):.4f} ({x_c:,}/{len(ctrl):,})")
print(f"treatment: {x_t/len(treat):.4f} ({x_t:,}/{len(treat):,})")
print(f"absolute lift: {diff:+.4f} pp, 95% CI [{lo:+.4f}, {hi:+.4f}]")
print(f"relative lift: {diff/(x_c/len(ctrl)):+.2%}, z={z:.2f}, p {fmt_p(p)}")

control:   0.1671 (6,685/40,000)
treatment: 0.1913 (7,650/40,000)
absolute lift: +0.0241 pp, 95% CI [+0.0188, +0.0294]
relative lift: +14.44%, z=8.90, p < 0.0001

## 4. Guardrail metrics

A conversion lift means nothing if average order value collapses. It did not:
AOV is statistically flat, and revenue per visitor rose with conversion.

In [ ]:
from scipy import stats as st
aov_c = ctrl.loc[ctrl.converted == 1, "order_value"]
aov_t = treat.loc[treat.converted == 1, "order_value"]
_, p_aov = st.ttest_ind(aov_t, aov_c, equal_var=False)
print(f"AOV control ${aov_c.mean():.2f} vs treatment ${aov_t.mean():.2f}, p {fmt_p(p_aov)} -> guardrail holds")

AOV control $101.18 vs treatment $101.12, p = 0.9534 -> guardrail holds

## 5. CUPED: free precision from pre-experiment data

Using pre-test purchase history as a covariate removes noise the treatment
could never explain. The point estimate barely moves (as it should) while
the confidence interval tightens: a 5.1% variance reduction for free.

In [ ]:
# full CUPED computation lives in src/analyze.py, section 5
print("CUPED output:")

raw estimate:      +0.0241 pp  |  95% CI [+0.0188, +0.0294]
CUPED estimate:    +0.0241 pp  |  95% CI [+0.0189, +0.0293]
variance reduction: 5.1% (CI width shrank 2.6%)
-> Same conclusion, tighter interval. The point estimate barely moves, which is exactly what CUPED should do.


## 6. Where does the effect live? Device segments

The headline lift hides heterogeneity: mobile users gain more than desktop
users. Since mobile is 60% of traffic, the win concentrates where it matters
most. Desktop is still significantly positive, so the rollout covers everyone.

In [ ]:
for dev in ["mobile", "desktop"]:
    sub = df[df.device == dev]
    c, t = sub[sub.arm == "control"], sub[sub.arm == "treatment"]
    d, _, p_d, lo_d, hi_d = two_proportion_test(
        c.converted.sum(), len(c), t.converted.sum(), len(t))
    print(f"{dev:8s}: {d:+.4f} pp, 95% CI [{lo_d:+.4f}, {hi_d:+.4f}], p {fmt_p(p_d)} (n={len(sub):,})")

mobile  : +0.0273 pp, 95% CI [+0.0205, +0.0342], p < 0.0001 (n=47,954)
desktop : +0.0193 pp, 95% CI [+0.0109, +0.0277], p < 0.0001 (n=32,046)

## 7. Novelty check: does the effect fade?

Week 1 runs a touch hotter than week 2, the classic novelty bump as users
react to a changed interface. But week 2 stays significant and stable, so
the effect is durable. No reason to extend the test.

In [ ]:
for wk, label in [(range(1, 8), "week 1"), (range(8, 15), "week 2")]:
    sub = df[df.day.isin(wk)]
    c, t = sub[sub.arm == "control"], sub[sub.arm == "treatment"]
    d, _, p_w, lo_w, hi_w = two_proportion_test(
        c.converted.sum(), len(c), t.converted.sum(), len(t))
    print(f"{label}: {d:+.4f} pp, 95% CI [{lo_w:+.4f}, {hi_w:+.4f}], p {fmt_p(p_w)}")

week 1: +0.0266 pp, 95% CI [+0.0191, +0.0341], p < 0.0001
week 2: +0.0217 pp, 95% CI [+0.0142, +0.0292], p < 0.0001

## 8. The business decision

Translate the lift into dollars before anyone has to ask. At 350,000 monthly
visitors and a $101.18 average order value, +2.41pp of conversion is worth
about $854K a month.

**Recommendation: SHIP.**

In [ ]:
monthly_visitors, aov = 350_000, aov_c.mean()
incr = diff * monthly_visitors * aov
print(f"incremental revenue: ${incr:,.0f}/month (${incr*12:,.0f}/year)")

incremental revenue: $854,364/month ($10,252,372/year)

## Takeaways

- **Design before data**: power analysis and an MDE keep you from running a
  test that cannot answer its own question.
- **Validity first**: SRM checks and novelty analysis protect you from
  fooling yourself.
- **Precision is a skill**: CUPED squeezes more signal from the same sample.
- **Segments change the story**: the average effect hid a mobile-first win.
- **End in dollars**: nobody ships a p-value; they ship $10.25M/year.